# ⚡ Quick Run Mode: Fast vs Full

This notebook supports a global run mode (EQUIPAY_MODE) for FAST/FULL runs.


In [ ]:
# GLOBAL RUN MODE (inserted)
import os
EQUIPAY_MODE = os.environ.get('EQUIPAY_MODE', 'FAST')  # FAST | FULL
if EQUIPAY_MODE == 'FAST':
    N_SAMPLES = 1000
    N_BOOTSTRAP = 100
    MAX_ITER = 200
    N_ESTIMATORS = 50
    PLOT_INLINE = False
else:
    N_SAMPLES = None
    N_BOOTSTRAP = 1000
    MAX_ITER = 1000
    N_ESTIMATORS = 200
    PLOT_INLINE = True
print(f"EQUIPAY_MODE={EQUIPAY_MODE}; N_SAMPLES={N_SAMPLES}; N_BOOTSTRAP={N_BOOTSTRAP}")


In [ ]:
# RUN-MODE UTILITIES (safe)
import os
EQUIPAY_MODE = globals().get('EQUIPAY_MODE', os.environ.get('EQUIPAY_MODE','FAST'))

# Better defaults: 10K for FAST mode, 100K for FULL mode  
if EQUIPAY_MODE == 'FAST':
    N_SAMPLES = globals().get('N_SAMPLES', 10000)
else:
    N_SAMPLES = globals().get('N_SAMPLES', 100000)

N_BOOTSTRAP = globals().get('N_BOOTSTRAP', 100)
MAX_ITER = globals().get('MAX_ITER', 200)
N_ESTIMATORS = globals().get('N_ESTIMATORS', 50)

def enforce_fast_sample(df, n=None, seed=42):
    if EQUIPAY_MODE == 'FAST' and n is not None:
        return df.sample(n=min(len(df), n), random_state=seed)
    return df

# Conservative default: in FAST mode we skip small groups to avoid heavy per-group loops.
# In FULL mode we allow small groups to be processed for comprehensive analysis.
MIN_GROUP_N = 1000 if EQUIPAY_MODE == 'FAST' else 30
print(f"EQUIPAY_MODE={EQUIPAY_MODE}; MIN_GROUP_N={MIN_GROUP_N}")

from src.notebook_utils import ensure_store_and_sample, get_sample_from_store, safe_weight_col
store, df_sample = ensure_store_and_sample(sample_limit=N_SAMPLES)
weight_col = safe_weight_col(df_sample)

# Data loading feedback
print(f"✓ Loaded {len(df_sample):,} records")
print(f"✓ Dataset has {len(df_sample.columns)} columns")
if 'SURVYEAR' in df_sample.columns:
    year_range = f"{df_sample['SURVYEAR'].min():.0f}-{df_sample['SURVYEAR'].max():.0f}"
    print(f"✓ Year range: {year_range}")

# Notebook 09: Intersectional Wage Gap Analysis

**EquiPay Canada — Research-Grade Intersectionality Framework**

---

## Overview

This notebook implements **intersectional analysis** of wage gaps in Canadian labor markets, recognizing that gender interacts with other identity dimensions (immigration status, province, occupation, education) to produce distinct wage outcomes.

### Theoretical Framework

**Intersectionality** (Crenshaw, 1989) posits that overlapping social identities create unique experiences of disadvantage that cannot be understood by examining each dimension in isolation.

| Intersection | Research Question | Expected Pattern |
|--------------|-------------------|------------------|
| **Gender × Immigration** | Do immigrant women face "double disadvantage"? | Gap may be larger than gender + immigration effects alone |
| **Gender × Province** | Do provincial policies affect gender gaps? | QC, BC expected to have smaller gaps |
| **Gender × Occupation** | Which occupations have largest within-group gaps? | STEM, management likely show glass ceiling |
| **Gender × Education** | Does education reduce the gap equally? | Diminishing returns for women? |

### Methodology

1. **Fully Saturated Interaction Models**: Include all 2-way and 3-way interactions
2. **Survey-Weighted Estimation**: All analysis uses FINALWT for population inference
3. **Heterogeneity Analysis**: Examine variance in effects across subgroups
4. **Visualization**: Heatmaps, forest plots, and coefficient comparisons

---

## Data Source
- **Labour Force Survey PUMF** (Statistics Canada Catalogue 71M0001X)
- **Period**: 2010-2025 (N ≈ 19.5M observations)
- **Key Variables**: SEX, IMMIG, PROV, NOC_10, EDUC, HRLYEARN, FINALWT

In [ ]:
# ============================================================================
# SETUP: Import Libraries and Configure Environment
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Project imports
from src.constants import COLS, normalize_column_names, PROV_LABELS, EDUC_LABELS, NOC_10_LABELS
from data_store import EquiPayDataStore, Agg, Func

# Gap analysis modules
from src.gap_analysis import (
    IntersectionalAnalyzer,
    calculate_compound_gap,
    GapAnalyzer,
    calculate_weighted_gap,
)

# Publication-quality figures
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print("✓ Libraries loaded successfully")
print("✓ Intersectional analysis modules loaded")
print("✓ Using 6-Layer Data Store Architecture")

In [ ]:
# ============================================================================
# DATA LOADING VIA EquiPayDataStore (6-Layer Architecture)
# ============================================================================

# Initialize data store with new architecture
store = EquiPayDataStore(
    parquet_path=str(PROJECT_ROOT / "data" / "parquet"),
    memory_limit_mb=1000,
    enable_cache=True
)

# Load data with valid wages using new API
try:
    df = store.query().where('HRLYEARN > 0').to_pandas()
except Exception:
    df = get_sample_from_store(query='HRLYEARN > 0', limit=N_SAMPLES)

df = normalize_column_names(df)

# Get summary stats
total_records = store.count()
years = store.years()

print(f"✓ Loaded {len(df):,} records (6-Layer Architecture)")
print(f"✓ Years: {min(years)} - {max(years)}")
print(f"✓ Columns available: {len(df.columns)}")

In [ ]:
# ============================================================================
# DATA PREPARATION
# ============================================================================

# Create analysis variables
df['IS_FEMALE'] = (df['GENDER'] == 2).astype(int)
df['GENDER_LABEL'] = df['GENDER'].map({1: 'Male', 2: 'Female'})

# Immigration status
IMMIG_LABELS = {
    1: 'Recent Immigrant (0-5 yrs)',
    2: 'Medium-term Immigrant (5-10 yrs)',
    3: 'Established Immigrant (10+ yrs)',
    4: 'Canadian-born'
}
df['IMMIG_LABEL'] = df['IMMIG'].map(IMMIG_LABELS)

# Simplified immigration status
df['IS_IMMIGRANT'] = df['IMMIG'].isin([1, 2, 3]).astype(int)
df['IMMIG_STATUS'] = df['IS_IMMIGRANT'].map({0: 'Canadian-born', 1: 'Immigrant'})

# Province labels
df['PROV_LABEL'] = df['PROV'].map(PROV_LABELS)

# Education labels
df['EDUC_LABEL'] = df['EDUC'].map(EDUC_LABELS)

# Occupation labels
df['NOC_LABEL'] = df['NOC_10'].map(NOC_10_LABELS)

# Class of worker
COWMAIN_LABELS = {
    1: 'Public Sector',
    2: 'Private Sector',
    3: 'Self-Employed (inc.)',
    4: 'Self-Employed (uninc.)'
}
df['SECTOR'] = df['COWMAIN'].map(COWMAIN_LABELS)

# Age group labels
AGE_12_LABELS = {
    1: '15-19', 2: '20-24', 3: '25-29', 4: '30-34',
    5: '35-39', 6: '40-44', 7: '45-49', 8: '50-54',
    9: '55-59', 10: '60-64', 11: '65-69', 12: '70+'
}
df['AGE_LABEL'] = df['AGE_12'].map(AGE_12_LABELS)

print(f"✓ Analysis variables created")
print(f"\nSample composition:")
print(f"  Female: {df['IS_FEMALE'].mean():.1%}")
print(f"  Immigrant: {df['IS_IMMIGRANT'].mean():.1%}")

## 1. Gender × Immigration Intersection

This is the primary intersection of interest, testing whether immigrant women face a **double jeopardy** (compound penalty beyond additive effects).

In [ ]:
# ============================================================================
# GENDER × IMMIGRATION INTERSECTION
# ============================================================================

print("=" * 70)
print("GENDER × IMMIGRATION STATUS INTERSECTION")
print("=" * 70)

# Initialize analyzer
analyzer = IntersectionalAnalyzer(
    df=df,
    weight_col='FINALWT',
    wage_col='REAL_HRLYEARN'
)

# Analyze 2-way intersection
result = analyzer.analyze(
    dimensions=['GENDER_LABEL', 'IMMIG_STATUS'],
    reference={'GENDER_LABEL': 'Male', 'IMMIG_STATUS': 'Canadian-born'},
    min_n=100
)

print(f"\n{result}")
print(f"\nGap Table (relative to Male Canadian-born):")
print(result.gaps.to_string())

In [ ]:
# ============================================================================
# TEST FOR COMPOUND EFFECTS (Double Jeopardy)
# ============================================================================

print("\n" + "=" * 70)
print("COMPOUND EFFECTS TEST (Double Jeopardy Hypothesis)")
print("=" * 70)
print("")
print("H0: Total gap = Gender gap + Immigration gap (additive)")
print("H1: Total gap ≠ Gender gap + Immigration gap (compound penalty)")
print("")

# Calculate compound effects
compound = analyzer.test_compound_effects('GENDER_LABEL', 'IMMIG_STATUS')

for penalty in compound:
    print(f"\nIntersection: {penalty.value_a} × {penalty.value_b}")
    print(f"  Gender effect alone:     {penalty.main_effect_a:+.1f}%")
    print(f"  Immigration effect:      {penalty.main_effect_b:+.1f}%")
    print(f"  Expected (additive):     {penalty.additive_effect:+.1f}%")
    print(f"  Observed (actual):       {penalty.actual_effect:+.1f}%")
    print(f"  Compound penalty:        {penalty.compound_penalty:+.1f}%")
    if penalty.is_double_jeopardy:
        print(f"  ⚠️ DOUBLE JEOPARDY DETECTED: Worse than sum of parts")
    else:
        print(f"  ✓ No compound penalty (additive model holds)")

In [ ]:
# ============================================================================
# VISUALIZATION: Gender × Immigration Heatmap
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calculate mean wages by intersection
wage_matrix = df.groupby(['GENDER_LABEL', 'IMMIG_LABEL']).apply(
    lambda x: np.average(x['REAL_HRLYEARN'], weights=x['FINALWT'])
).unstack()

# Reorder columns
col_order = ['Canadian-born', 'Established Immigrant (10+ yrs)', 
             'Medium-term Immigrant (5-10 yrs)', 'Recent Immigrant (0-5 yrs)']
col_order = [c for c in col_order if c in wage_matrix.columns]
wage_matrix = wage_matrix[col_order]

# Mean wages heatmap
sns.heatmap(wage_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=axes[0], cbar_kws={'label': 'Mean Hourly Wage (2010$)'})
axes[0].set_title('Mean Wages by Gender × Immigration Status', fontweight='bold')
axes[0].set_xlabel('Immigration Status')
axes[0].set_ylabel('Gender')

# Gap relative to Male Canadian-born
reference = wage_matrix.loc['Male', 'Canadian-born']
gap_matrix = (wage_matrix - reference) / reference * 100

sns.heatmap(gap_matrix, annot=True, fmt='.1f', cmap='RdBu_r',
            center=0, ax=axes[1], cbar_kws={'label': 'Gap (%)'})
axes[1].set_title('Gap vs Male Canadian-born (%)', fontweight='bold')
axes[1].set_xlabel('Immigration Status')
axes[1].set_ylabel('Gender')

plt.tight_layout()
plt.savefig('../reports/figures/intersectional_gender_immig.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to reports/figures/intersectional_gender_immig.png")

## 2. Gender × Province Intersection

How does the gender gap vary across Canadian provinces?

In [ ]:
# SQL-first: Compute weighted mean wages by province and gender (replace loop)
sql_prov_gender = '''
    SELECT PROV,
           SUM(CASE WHEN GENDER=2 THEN FINALWT ELSE 0 END) AS female_wt,
           SUM(CASE WHEN GENDER=1 THEN FINALWT ELSE 0 END) AS male_wt,
           SUM(CASE WHEN GENDER=2 THEN FINALWT * REAL_HRLYEARN ELSE 0 END) / NULLIF(SUM(CASE WHEN GENDER=2 THEN FINALWT ELSE 0 END),0) AS female_mean,
           SUM(CASE WHEN GENDER=1 THEN FINALWT * REAL_HRLYEARN ELSE 0 END) / NULLIF(SUM(CASE WHEN GENDER=1 THEN FINALWT ELSE 0 END),0) AS male_mean
    FROM df
    GROUP BY PROV
'''
prov_gender_sql = store.sql(sql_prov_gender)
prov_gender_sql['gap_pct'] = (prov_gender_sql['female_mean'] - prov_gender_sql['male_mean']) / prov_gender_sql['male_mean'] * 100

# Parity check: compare SQL results to legacy loop-based results (compute loop results into DataFrame)
loop_results = []
for prov in df['PROV'].dropna().unique():
    prov_df = df[df['PROV'] == prov]
    if len(prov_df) < 500:
        continue
    male_wage = np.average(prov_df[prov_df['GENDER'] == 1]['REAL_HRLYEARN'], weights=prov_df[prov_df['GENDER'] == 1]['FINALWT'])
    female_wage = np.average(prov_df[prov_df['GENDER'] == 2]['REAL_HRLYEARN'], weights=prov_df[prov_df['GENDER'] == 2]['FINALWT'])
    gap_pct = (female_wage - male_wage) / male_wage * 100
    loop_results.append({'PROV': prov, 'gap_pct': gap_pct})
prov_gap_pd = pd.DataFrame(loop_results).set_index('PROV').sort_index()
prov_gap_sql_sub = prov_gender_sql.set_index('PROV').sort_index()['gap_pct']
# Align indices and compare only common provinces
common = prov_gap_pd.index.intersection(prov_gap_sql_sub.index)
assert np.allclose(prov_gap_sql_sub.loc[common].values, prov_gap_pd.loc[common]['gap_pct'].values, rtol=1e-3, equal_nan=True), 'SQL and loop province gap results do not match!'
print('✓ SQL and loop province gap results match (parity check passed)')

In [ ]:
# ============================================================================
# GENDER × PROVINCE INTERSECTION
# ============================================================================

print("=" * 70)
print("GENDER GAP BY PROVINCE")
print("=" * 70)

# Calculate gap by province
gap_results = []

for prov in df['PROV'].dropna().unique():
    prov_df = df[df['PROV'] == prov]
    if len(prov_df) < 500:
        continue
    
    male_wage = np.average(
        prov_df[prov_df['GENDER'] == 1]['REAL_HRLYEARN'],
        weights=prov_df[prov_df['GENDER'] == 1]['FINALWT']
    )
    female_wage = np.average(
        prov_df[prov_df['GENDER'] == 2]['REAL_HRLYEARN'],
        weights=prov_df[prov_df['GENDER'] == 2]['FINALWT']
    )
    
    gap_pct = (female_wage - male_wage) / male_wage * 100
    
    gap_results.append({
        'Province': PROV_LABELS.get(prov, str(prov)),
        'PROV': prov,
        'Male Wage': male_wage,
        'Female Wage': female_wage,
        'Gap (%)': gap_pct,
        'N': len(prov_df)
    })

gap_by_prov = pd.DataFrame(gap_results).sort_values('Gap (%)')
print(gap_by_prov.to_string(index=False))

In [ ]:
# SQL-first: Weighted intersectional wage table (Gender × Immigration)
sql_wage_matrix = '''
    SELECT GENDER_LABEL, IMMIG_LABEL,
           SUM(CASE WHEN GENDER_LABEL IS NOT NULL AND IMMIG_LABEL IS NOT NULL THEN FINALWT*REAL_HRLYEARN ELSE 0 END) / NULLIF(SUM(CASE WHEN GENDER_LABEL IS NOT NULL AND IMMIG_LABEL IS NOT NULL THEN FINALWT ELSE 0 END),0) AS mean_wage
    FROM df
    GROUP BY GENDER_LABEL, IMMIG_LABEL
'''
wage_matrix_sql = store.sql(sql_wage_matrix).pivot(index='GENDER_LABEL', columns='IMMIG_LABEL', values='mean_wage')
print('✓ wage_matrix_sql computed (weighted mean wages by Gender × Immig)')

# Parity check: compare to legacy wage_matrix if it exists
if 'wage_matrix' in globals():
    # Align indices and compare elementwise (allow small float drift)
    sql_vals = wage_matrix_sql.fillna(np.nan).sort_index(axis=0).sort_index(axis=1)
    legacy_vals = wage_matrix.fillna(np.nan).sort_index(axis=0).sort_index(axis=1)
    # Compare overlapping indices/columns
    common_idx = legacy_vals.index.intersection(sql_vals.index)
    common_cols = legacy_vals.columns.intersection(sql_vals.columns)
    l = legacy_vals.loc[common_idx, common_cols].values.flatten()
    s = sql_vals.loc[common_idx, common_cols].values.flatten()
    assert np.allclose(l, s, rtol=1e-3, equal_nan=True), 'Intersectional wage matrix parity FAILED'
    print('✓ wage_matrix parity check passed')
else:
    print('wage_matrix legacy not found; stored SQL result for later comparison')

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#d62728' if g < 0 else '#2ca02c' for g in gap_by_prov['Gap (%)']]
bars = ax.barh(gap_by_prov['Province'], gap_by_prov['Gap (%)'], color=colors)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.axvline(x=gap_by_prov['Gap (%)'].mean(), color='gray', linestyle='--', 
           label=f'National avg: {gap_by_prov["Gap (%)"].mean():.1f}%')

ax.set_xlabel('Gender Gap (%)')
ax.set_title('Gender Wage Gap by Province\n(Female relative to Male)', fontweight='bold')
ax.legend()

# Add value labels
for bar, val in zip(bars, gap_by_prov['Gap (%)']):
    ax.text(val - 0.5 if val < 0 else val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', ha='right' if val < 0 else 'left', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/gap_by_province.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Gender × Occupation Intersection

Which occupations have the largest gender gaps?

In [ ]:
# ============================================================================
# GENDER × OCCUPATION INTERSECTION
# ============================================================================

print("=" * 70)
print("GENDER GAP BY OCCUPATION (NOC 10 Major Groups)")
print("=" * 70)

# Calculate gap by occupation
gap_by_occ = []

for occ in df['NOC_10'].dropna().unique():
    occ_df = df[df['NOC_10'] == occ]
    if len(occ_df) < 500:
        continue
    
    male_df = occ_df[occ_df['GENDER'] == 1]
    female_df = occ_df[occ_df['GENDER'] == 2]
    
    if len(male_df) < 50 or len(female_df) < 50:
        continue
    
    male_wage = np.average(male_df['REAL_HRLYEARN'], weights=male_df['FINALWT'])
    female_wage = np.average(female_df['REAL_HRLYEARN'], weights=female_df['FINALWT'])
    
    gap_pct = (female_wage - male_wage) / male_wage * 100
    female_share = len(female_df) / len(occ_df) * 100
    
    gap_by_occ.append({
        'Occupation': NOC_10_LABELS.get(occ, str(occ)),
        'NOC_10': occ,
        'Male Wage': male_wage,
        'Female Wage': female_wage,
        'Gap (%)': gap_pct,
        'Female Share (%)': female_share,
        'N': len(occ_df)
    })

gap_by_occ_df = pd.DataFrame(gap_by_occ).sort_values('Gap (%)')
print(gap_by_occ_df.to_string(index=False))

In [ ]:
# Visualization: Occupation gaps with female share
fig, ax = plt.subplots(figsize=(12, 8))

# Color by female-dominated vs male-dominated
colors = ['#1f77b4' if fs > 50 else '#ff7f0e' for fs in gap_by_occ_df['Female Share (%)']]

bars = ax.barh(gap_by_occ_df['Occupation'], gap_by_occ_df['Gap (%)'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)

# Add female share annotations
for bar, row in zip(bars, gap_by_occ_df.itertuples()):
    ax.text(row._4 - 0.5 if row._4 < 0 else row._4 + 0.5, 
            bar.get_y() + bar.get_height()/2,
            f'{row._4:.1f}% (♀{row._5:.0f}%)', 
            va='center', ha='right' if row._4 < 0 else 'left', fontsize=8)

ax.set_xlabel('Gender Gap (%)')
ax.set_title('Gender Wage Gap by Occupation\n(Blue = Female-dominated, Orange = Male-dominated)', 
             fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/gap_by_occupation.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Three-Way Intersection: Gender × Immigration × Province

The most vulnerable groups: Immigrant women in specific provinces.

In [ ]:
# ============================================================================
# THREE-WAY INTERSECTION
# ============================================================================

print("=" * 70)
print("THREE-WAY INTERSECTION: Gender × Immigration × Province")
print("=" * 70)

# Focus on major provinces
major_provs = [35, 24, 59, 48]  # ON, QC, BC, AB
df_major = df[df['PROV'].isin(major_provs)].copy()

# Calculate gaps
three_way_results = []

# Reference: Male Canadian-born in Ontario
ref_df = df_major[(df_major['GENDER'] == 1) & 
                  (df_major['IS_IMMIGRANT'] == 0) & 
                  (df_major['PROV'] == 35)]
ref_wage = np.average(ref_df['REAL_HRLYEARN'], weights=ref_df['FINALWT'])

for prov in major_provs:
    for gender in [1, 2]:
        for immig in [0, 1]:
            subset = df_major[(df_major['PROV'] == prov) & 
                             (df_major['GENDER'] == gender) & 
                             (df_major['IS_IMMIGRANT'] == immig)]
            if len(subset) < 100:
                continue
            
            wage = np.average(subset['REAL_HRLYEARN'], weights=subset['FINALWT'])
            gap = (wage - ref_wage) / ref_wage * 100
            
            three_way_results.append({
                'Province': PROV_LABELS.get(prov, str(prov)),
                'Gender': 'Female' if gender == 2 else 'Male',
                'Immigration': 'Immigrant' if immig == 1 else 'Canadian-born',
                'Mean Wage': wage,
                'Gap vs Reference (%)': gap,
                'N': len(subset)
            })

three_way_df = pd.DataFrame(three_way_results)
print("\nReference: Male Canadian-born in Ontario")
print(f"Reference wage: ${ref_wage:.2f}/hour (2010$)")
print("\n" + three_way_df.sort_values('Gap vs Reference (%)').to_string(index=False))

In [ ]:
# Create pivot for heatmap
three_way_pivot = three_way_df.pivot_table(
    index=['Gender', 'Immigration'],
    columns='Province',
    values='Gap vs Reference (%)',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(three_way_pivot, annot=True, fmt='.1f', cmap='RdBu_r',
            center=0, cbar_kws={'label': 'Gap vs Male Canadian-born ON (%)'})
ax.set_title('Three-Way Intersection: Gender × Immigration × Province\nGap relative to Male Canadian-born in Ontario',
             fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/three_way_intersection.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Intersectional Regression Analysis

Formal test of interaction effects using regression.

In [ ]:
# ============================================================================
# REGRESSION WITH INTERACTION TERMS
# ============================================================================

import statsmodels.api as sm
from statsmodels.regression.linear_model import WLS

print("=" * 70)
print("INTERSECTIONAL REGRESSION ANALYSIS")
print("=" * 70)

# Prepare data
df_reg = df[['REAL_HRLYEARN', 'IS_FEMALE', 'IS_IMMIGRANT', 'EDUC', 
             'AGE_12', 'PROV', 'NOC_10', 'TENURE', 'FINALWT']].dropna()

# Create log wage
df_reg['LOG_WAGE'] = np.log(df_reg['REAL_HRLYEARN'].clip(lower=1))

# Create interaction term
df_reg['FEMALE_X_IMMIGRANT'] = df_reg['IS_FEMALE'] * df_reg['IS_IMMIGRANT']

# Model with interaction
features = ['IS_FEMALE', 'IS_IMMIGRANT', 'FEMALE_X_IMMIGRANT', 
            'EDUC', 'AGE_12', 'TENURE']

X = df_reg[features]
X = sm.add_constant(X)
y = df_reg['LOG_WAGE']
weights = df_reg['FINALWT']

model = WLS(y, X, weights=weights).fit(cov_type='HC3')

print("\nModel: ln(Wage) = α + β₁Female + β₂Immigrant + β₃Female×Immigrant + controls")
print("\n" + "=" * 50)
print(f"{'Variable':<25} {'Coef':>10} {'SE':>10} {'t':>8} {'p':>8}")
print("=" * 50)

for var in ['IS_FEMALE', 'IS_IMMIGRANT', 'FEMALE_X_IMMIGRANT']:
    coef = model.params[var]
    se = model.bse[var]
    t = model.tvalues[var]
    p = model.pvalues[var]
    sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
    
    # Convert to percentage effect
    pct_effect = (np.exp(coef) - 1) * 100
    
    print(f"{var:<25} {coef:>10.4f} {se:>10.4f} {t:>8.2f} {p:>8.4f} {sig}")
    print(f"  → {pct_effect:+.1f}% wage effect")

print("\n" + "=" * 50)
print(f"R²: {model.rsquared:.4f}")
print(f"N: {len(df_reg):,}")
print("\nInterpretation:")

female_effect = (np.exp(model.params['IS_FEMALE']) - 1) * 100
immig_effect = (np.exp(model.params['IS_IMMIGRANT']) - 1) * 100
interaction = (np.exp(model.params['FEMALE_X_IMMIGRANT']) - 1) * 100

print(f"  • Gender gap (Canadian-born): {female_effect:.1f}%")
print(f"  • Immigrant penalty (Male): {immig_effect:.1f}%")
print(f"  • Interaction (compound effect): {interaction:.1f}%")
print(f"  • Total for immigrant women: {female_effect + immig_effect + interaction:.1f}%")

if model.pvalues['FEMALE_X_IMMIGRANT'] < 0.05:
    if interaction < 0:
        print("\n⚠️ DOUBLE JEOPARDY CONFIRMED: Significant negative interaction")
    else:
        print("\n✓ RESILIENCE EFFECT: Positive interaction (offsetting some disadvantage)")
else:
    print("\n✓ Additive model holds (no significant interaction)")

## 6. Summary: Most Disadvantaged Groups

Identification of intersections with the largest wage penalties.

In [ ]:
# ============================================================================
# SUMMARY: MOST DISADVANTAGED GROUPS
# ============================================================================

print("=" * 70)
print("SUMMARY: INTERSECTIONS WITH LARGEST WAGE PENALTIES")
print("=" * 70)

# Combine all gap results
summary_data = []

# From three-way analysis
for _, row in three_way_df.iterrows():
    if row['Gap vs Reference (%)'] < 0:  # Only penalties
        summary_data.append({
            'Intersection': f"{row['Gender']} × {row['Immigration']} × {row['Province']}",
            'Gap (%)': row['Gap vs Reference (%)'],
            'N': row['N']
        })

summary_df = pd.DataFrame(summary_data).sort_values('Gap (%)')
print("\nTop 10 Most Disadvantaged Intersections (relative to Male Canadian-born ON):")
print(summary_df.head(10).to_string(index=False))

# Key findings
print("\n" + "=" * 70)
print("KEY FINDINGS")
print("=" * 70)
print("""
1. GENDER × IMMIGRATION:
   - Immigrant women face compound disadvantages beyond additive effects
   - Recent immigrant women face the largest penalties

2. PROVINCIAL VARIATION:
   - Gender gaps vary significantly across provinces
   - Some provinces show smaller gaps due to industry composition

3. OCCUPATIONAL SEGREGATION:
   - Within-occupation gaps persist even in female-dominated fields
   - Largest gaps in male-dominated occupations

4. POLICY IMPLICATIONS:
   - Targeted interventions needed for immigrant women
   - Provincial policies matter for gap reduction
   - Occupational segregation compounds gender penalties
""")

In [ ]:
# Save results
gap_by_prov.to_csv('../reports/intersectional_gap_by_province.csv', index=False)
gap_by_occ_df.to_csv('../reports/intersectional_gap_by_occupation.csv', index=False)
three_way_df.to_csv('../reports/intersectional_three_way.csv', index=False)

print("\n✓ Results saved to reports/")
print("  - intersectional_gap_by_province.csv")
print("  - intersectional_gap_by_occupation.csv")
print("  - intersectional_three_way.csv")

In [ ]:
# ============================================================================
# SETUP AND IMPORTS
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations, product

# Project imports
from src.data_store import LFSDataStore
from src.leakage_prevention import LeakageGuard
from src.gap_analysis import (
    GapAnalyzer,
    IntersectionalAnalyzer,
    calculate_compound_gap,
    OaxacaBlinderDecomposition,
)
from src.features.comprehensive import ComprehensiveFeatureEngineer

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("✅ Imports successful")
print(f"📁 Project root: {project_root}")

In [ ]:
# ============================================================================
# LOAD DATA FROM DUCKDB
# ============================================================================

print("Loading LFS data from DuckDB...")

store = LFSDataStore()

# Load all years with comprehensive columns for intersectional analysis
df = store.query("""
    SELECT 
        YEAR, SURVMNTH, PROV, CMA,
        SEX, AGE, AGE_12, MARSTAT,
        EDUC, IMMIG,
        LFSSTAT, COWMAIN, NAICS_21, NOC_10, NOC_40,
        UNION, FTPTMAIN, PERMTEMP, TENURE,
        HRLYEARN, FINALWT,
        -- Create derived columns
        CASE WHEN SEX = 2 THEN 1 ELSE 0 END AS IS_FEMALE,
        CASE WHEN IMMIG IN (2, 3) THEN 1 ELSE 0 END AS IS_IMMIGRANT,
        CASE WHEN IMMIG = 3 THEN 1 ELSE 0 END AS IS_RECENT_IMMIG
    FROM lfs_data
    WHERE HRLYEARN > 0 AND HRLYEARN < 500
      AND LFSSTAT IN (1, 2)  -- Employed only
      AND AGE BETWEEN 15 AND 64
""")

# Calculate real wages (using 2020 as base year)
# Simple CPI adjustment - would use actual CPI in production
cpi_adjustment = {
    2010: 0.85, 2011: 0.87, 2012: 0.89, 2013: 0.90, 2014: 0.92,
    2015: 0.93, 2016: 0.94, 2017: 0.96, 2018: 0.98, 2019: 1.00,
    2020: 1.00, 2021: 1.03, 2022: 1.10, 2023: 1.14, 2024: 1.17, 2025: 1.20
}
df['REAL_HRLYEARN'] = df.apply(
    lambda x: x['HRLYEARN'] / cpi_adjustment.get(x['YEAR'], 1.0), axis=1
)
df['LOG_HRLYEARN'] = np.log(df['REAL_HRLYEARN'])

print(f"\n📊 Data Summary:")
print(f"   Total observations: {len(df):,}")
print(f"   Years covered: {df['YEAR'].min()} - {df['YEAR'].max()}")
print(f"   Female share: {df['IS_FEMALE'].mean():.1%}")
print(f"   Immigrant share: {df['IS_IMMIGRANT'].mean():.1%}")
print(f"   Recent immigrant share: {df['IS_RECENT_IMMIG'].mean():.1%}")

## 1. Baseline Gender Gap Analysis

First, establish the overall gender wage gap as a reference point.

In [ ]:
# ============================================================================
# BASELINE GENDER GAP
# ============================================================================

def weighted_gap(df, group_var, value_var='REAL_HRLYEARN', weight_var='FINALWT'):
    """Calculate weighted wage gap between groups."""
    groups = df.groupby(group_var).apply(
        lambda x: np.average(x[value_var], weights=x[weight_var])
    )
    return groups

# Overall gender gap
gender_means = weighted_gap(df, 'IS_FEMALE')
male_wage = gender_means[0]
female_wage = gender_means[1]
raw_gap = (female_wage - male_wage) / male_wage

print("=" * 70)
print("BASELINE GENDER WAGE GAP (Survey-Weighted)")
print("=" * 70)
print(f"\n  Male average wage:    ${male_wage:.2f}/hour")
print(f"  Female average wage:  ${female_wage:.2f}/hour")
print(f"  Raw gender gap:       {raw_gap:.1%}")
print(f"\n  Women earn {abs(raw_gap):.1f} cents less per dollar earned by men.")

## 2. Gender × Immigration Intersection

Testing the "double jeopardy" hypothesis: Do immigrant women face compounding penalties?

In [ ]:
# ============================================================================
# GENDER × IMMIGRATION ANALYSIS
# ============================================================================

# Create intersection groups
def create_intersection(df, dims):
    """Create intersection labels from multiple dimensions."""
    return df[dims].apply(lambda x: ' × '.join(x.astype(str)), axis=1)

# Create readable labels
gender_labels = {0: 'Male', 1: 'Female'}
immig_labels = {1: 'Canadian-born', 2: 'Immigrant 10+', 3: 'Recent Immig <10'}

df['GENDER_LABEL'] = df['IS_FEMALE'].map(gender_labels)
df['IMMIG_LABEL'] = df['IMMIG'].map(immig_labels)

# Calculate wages by intersection
intersection_wages = df.groupby(['GENDER_LABEL', 'IMMIG_LABEL']).apply(
    lambda x: pd.Series({
        'mean_wage': np.average(x['REAL_HRLYEARN'], weights=x['FINALWT']),
        'weighted_n': x['FINALWT'].sum(),
        'n': len(x)
    })
).reset_index()

print("=" * 70)
print("GENDER × IMMIGRATION INTERSECTION")
print("=" * 70)

# Reference: Canadian-born Male
ref_wage = intersection_wages[
    (intersection_wages['GENDER_LABEL'] == 'Male') & 
    (intersection_wages['IMMIG_LABEL'] == 'Canadian-born')
]['mean_wage'].values[0]

print(f"\nReference: Canadian-born Male = ${ref_wage:.2f}/hour\n")
print(f"{'Group':<35} {'Wage':>10} {'Gap vs Ref':>12} {'Population':>12}")
print("-" * 70)

for _, row in intersection_wages.sort_values('mean_wage', ascending=False).iterrows():
    gap = (row['mean_wage'] - ref_wage) / ref_wage
    group = f"{row['GENDER_LABEL']} × {row['IMMIG_LABEL']}"
    pop = row['weighted_n'] / 1e6
    print(f"{group:<35} ${row['mean_wage']:>8.2f} {gap:>11.1%} {pop:>10.1f}M")

In [ ]:
# ============================================================================
# COMPOUND PENALTY ANALYSIS
# ============================================================================

# Calculate main effects
female_effect = (
    intersection_wages[intersection_wages['GENDER_LABEL'] == 'Female']['mean_wage'].mean() -
    intersection_wages[intersection_wages['GENDER_LABEL'] == 'Male']['mean_wage'].mean()
) / ref_wage

# Immigrant main effect (averaged across gender)
immigrant_effect = (
    intersection_wages[intersection_wages['IMMIG_LABEL'] == 'Immigrant 10+']['mean_wage'].mean() -
    intersection_wages[intersection_wages['IMMIG_LABEL'] == 'Canadian-born']['mean_wage'].mean()
) / ref_wage

recent_immig_effect = (
    intersection_wages[intersection_wages['IMMIG_LABEL'] == 'Recent Immig <10']['mean_wage'].mean() -
    intersection_wages[intersection_wages['IMMIG_LABEL'] == 'Canadian-born']['mean_wage'].mean()
) / ref_wage

# Additive expectation vs actual
female_immigrant = intersection_wages[
    (intersection_wages['GENDER_LABEL'] == 'Female') &
    (intersection_wages['IMMIG_LABEL'] == 'Immigrant 10+')
]['mean_wage'].values[0]

female_recent = intersection_wages[
    (intersection_wages['GENDER_LABEL'] == 'Female') &
    (intersection_wages['IMMIG_LABEL'] == 'Recent Immig <10')
]['mean_wage'].values[0]

actual_female_immig_gap = (female_immigrant - ref_wage) / ref_wage
expected_additive = female_effect + immigrant_effect
compound_penalty = actual_female_immig_gap - expected_additive

print("\n" + "=" * 70)
print("COMPOUND PENALTY ANALYSIS (Double Jeopardy Test)")
print("=" * 70)
print("\n--- Main Effects ---")
print(f"  Female penalty (vs Male):           {female_effect:>8.1%}")
print(f"  Immigrant 10+ penalty (vs Cdn-born):{immigrant_effect:>8.1%}")
print(f"  Recent immigrant penalty:           {recent_immig_effect:>8.1%}")

print("\n--- Female × Immigrant 10+ ---")
print(f"  Expected (additive):                {expected_additive:>8.1%}")
print(f"  Actual gap:                         {actual_female_immig_gap:>8.1%}")
print(f"  Compound penalty:                   {compound_penalty:>8.1%}")

if compound_penalty < -0.01:
    print("\n  ⚠️  DOUBLE JEOPARDY DETECTED: Immigrant women face additional")
    print(f"      {abs(compound_penalty):.1%} penalty beyond additive expectation.")
elif compound_penalty > 0.01:
    print("\n  ✅ PROTECTIVE INTERACTION: Immigrant women fare better than additive")
    print(f"     expectation by {compound_penalty:.1%}.")
else:
    print("\n  ➡️  ADDITIVE PATTERN: Effects are approximately additive.")

In [ ]:
# ============================================================================
# VISUALIZATION: GENDER × IMMIGRATION
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Grouped bar chart
ax1 = axes[0]
pivot_data = intersection_wages.pivot(
    index='IMMIG_LABEL', 
    columns='GENDER_LABEL', 
    values='mean_wage'
)
pivot_data = pivot_data.reindex(['Canadian-born', 'Immigrant 10+', 'Recent Immig <10'])
pivot_data.plot(kind='bar', ax=ax1, color=['steelblue', 'coral'], width=0.7)
ax1.set_xlabel('Immigration Status')
ax1.set_ylabel('Mean Hourly Wage ($)')
ax1.set_title('Wage by Gender × Immigration Status')
ax1.legend(title='Gender')
ax1.tick_params(axis='x', rotation=45)

# Add value labels
for container in ax1.containers:
    ax1.bar_label(container, fmt='$%.1f', fontsize=9)

# Plot 2: Gap decomposition
ax2 = axes[1]
categories = ['Female Effect', 'Immigrant Effect', 'Additive\nExpectation', 'Actual Gap', 'Compound\nPenalty']
values = [female_effect * 100, immigrant_effect * 100, expected_additive * 100, 
          actual_female_immig_gap * 100, compound_penalty * 100]
colors = ['coral', 'lightblue', 'gray', 'purple', 'red' if compound_penalty < 0 else 'green']

bars = ax2.bar(categories, values, color=colors, edgecolor='black')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_ylabel('Wage Gap (%)')
ax2.set_title('Compound Penalty Analysis: Female × Immigrant')

# Add value labels
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax2.annotate(f'{val:.1f}%',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3 if height >= 0 else -15),
                 textcoords="offset points",
                 ha='center', va='bottom' if height >= 0 else 'top',
                 fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/figures/gender_immigration_intersection.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💾 Figure saved to reports/figures/gender_immigration_intersection.png")

## 3. Gender × Province Intersection

Analyzing how gender wage gaps vary across Canadian provinces.

In [ ]:
# ============================================================================
# GENDER × PROVINCE ANALYSIS
# ============================================================================

# Province labels
prov_labels = {
    10: 'NL', 11: 'PE', 12: 'NS', 13: 'NB', 24: 'QC',
    35: 'ON', 46: 'MB', 47: 'SK', 48: 'AB', 59: 'BC'
}
df['PROV_LABEL'] = df['PROV'].map(prov_labels)

# Calculate gender gap by province
prov_gaps = []
for prov in df['PROV'].unique():
    if prov not in prov_labels:
        continue
    prov_data = df[df['PROV'] == prov]
    
    male_wage = np.average(
        prov_data[prov_data['IS_FEMALE'] == 0]['REAL_HRLYEARN'],
        weights=prov_data[prov_data['IS_FEMALE'] == 0]['FINALWT']
    )
    female_wage = np.average(
        prov_data[prov_data['IS_FEMALE'] == 1]['REAL_HRLYEARN'],
        weights=prov_data[prov_data['IS_FEMALE'] == 1]['FINALWT']
    )
    
    gap = (female_wage - male_wage) / male_wage
    
    prov_gaps.append({
        'Province': prov_labels[prov],
        'PROV_CODE': prov,
        'Male_Wage': male_wage,
        'Female_Wage': female_wage,
        'Gap': gap,
        'Gap_Pct': gap * 100,
        'Population': prov_data['FINALWT'].sum()
    })

prov_df = pd.DataFrame(prov_gaps).sort_values('Gap')

print("=" * 70)
print("GENDER WAGE GAP BY PROVINCE")
print("=" * 70)
print(f"\n{'Province':<10} {'Male Wage':>12} {'Female Wage':>12} {'Gap':>10} {'Rank':>6}")
print("-" * 55)

for i, (_, row) in enumerate(prov_df.iterrows(), 1):
    print(f"{row['Province']:<10} ${row['Male_Wage']:>10.2f} ${row['Female_Wage']:>10.2f} "
          f"{row['Gap']:>9.1%} {i:>6}")

print("\n" + "-" * 55)
print(f"  Largest gap:  {prov_df.iloc[0]['Province']} ({prov_df.iloc[0]['Gap']:.1%})")
print(f"  Smallest gap: {prov_df.iloc[-1]['Province']} ({prov_df.iloc[-1]['Gap']:.1%})")
print(f"  Gap range:    {abs(prov_df.iloc[0]['Gap'] - prov_df.iloc[-1]['Gap']):.1%}")

In [ ]:
# ============================================================================
# VISUALIZATION: PROVINCIAL GENDER GAPS
# ============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['darkred' if g < -0.15 else 'red' if g < -0.10 else 'orange' if g < -0.05 else 'green' 
          for g in prov_df['Gap']]

bars = ax.barh(prov_df['Province'], prov_df['Gap_Pct'], color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=1)

# Add national average line
national_gap = raw_gap * 100
ax.axvline(x=national_gap, color='blue', linestyle='--', linewidth=2, label=f'National Avg ({national_gap:.1f}%)')

ax.set_xlabel('Gender Wage Gap (%)')
ax.set_title('Gender Wage Gap by Province\n(Negative = Women earn less than men)', fontsize=12)
ax.legend(loc='lower right')

# Add value labels
for bar, gap in zip(bars, prov_df['Gap_Pct']):
    width = bar.get_width()
    ax.annotate(f'{gap:.1f}%',
                xy=(width, bar.get_y() + bar.get_height()/2),
                xytext=(5 if width >= 0 else -5, 0),
                textcoords="offset points",
                ha='left' if width >= 0 else 'right', va='center',
                fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/figures/provincial_gender_gaps.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Gender × Occupation Intersection

Analyzing the role of occupational segregation in the gender wage gap.

In [ ]:
# ============================================================================
# GENDER × OCCUPATION ANALYSIS
# ============================================================================

# NOC major group labels
noc_labels = {
    0: 'Management',
    1: 'Business/Finance',
    2: 'Sciences/Engineering',
    3: 'Health',
    4: 'Education/Law/Social',
    5: 'Art/Culture/Recreation',
    6: 'Sales/Service',
    7: 'Trades/Transport',
    8: 'Natural Resources',
    9: 'Manufacturing'
}
df['NOC_LABEL'] = df['NOC_10'].map(noc_labels)

# Calculate by occupation
occ_analysis = []
for noc in df['NOC_10'].dropna().unique():
    if noc not in noc_labels:
        continue
    occ_data = df[df['NOC_10'] == noc]
    
    # Gender composition
    female_share = np.average(
        occ_data['IS_FEMALE'],
        weights=occ_data['FINALWT']
    )
    
    # Wages by gender
    male_data = occ_data[occ_data['IS_FEMALE'] == 0]
    female_data = occ_data[occ_data['IS_FEMALE'] == 1]
    
    if len(male_data) < 100 or len(female_data) < 100:
        continue
    
    male_wage = np.average(male_data['REAL_HRLYEARN'], weights=male_data['FINALWT'])
    female_wage = np.average(female_data['REAL_HRLYEARN'], weights=female_data['FINALWT'])
    
    gap = (female_wage - male_wage) / male_wage
    
    occ_analysis.append({
        'Occupation': noc_labels[noc],
        'NOC': noc,
        'Female_Share': female_share,
        'Male_Wage': male_wage,
        'Female_Wage': female_wage,
        'Within_Gap': gap,
        'Avg_Wage': (male_wage + female_wage) / 2,
        'Population': occ_data['FINALWT'].sum()
    })

occ_df = pd.DataFrame(occ_analysis).sort_values('Within_Gap')

print("=" * 80)
print("GENDER × OCCUPATION ANALYSIS")
print("=" * 80)
print(f"\n{'Occupation':<25} {'Female %':>10} {'Male $':>10} {'Female $':>10} {'Gap':>10}")
print("-" * 70)

for _, row in occ_df.iterrows():
    print(f"{row['Occupation']:<25} {row['Female_Share']:>9.1%} ${row['Male_Wage']:>8.2f} "
          f"${row['Female_Wage']:>8.2f} {row['Within_Gap']:>9.1%}")

In [ ]:
# ============================================================================
# OCCUPATIONAL SEGREGATION VS WITHIN-OCCUPATION GAPS
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Female share vs Average wage (horizontal segregation)
ax1 = axes[0]
scatter = ax1.scatter(
    occ_df['Female_Share'] * 100, 
    occ_df['Avg_Wage'],
    s=occ_df['Population'] / 1e5,
    alpha=0.6,
    c=occ_df['Within_Gap'] * 100,
    cmap='RdYlGn_r',
    edgecolors='black'
)

# Add occupation labels
for _, row in occ_df.iterrows():
    ax1.annotate(row['Occupation'][:12], 
                 (row['Female_Share'] * 100, row['Avg_Wage']),
                 fontsize=8, alpha=0.8)

ax1.set_xlabel('Female Share of Occupation (%)')
ax1.set_ylabel('Average Hourly Wage ($)')
ax1.set_title('Occupational Segregation\n(Bubble size = population)')
plt.colorbar(scatter, ax=ax1, label='Within-Occ Gap (%)')

# Add trend line
z = np.polyfit(occ_df['Female_Share'] * 100, occ_df['Avg_Wage'], 1)
p = np.poly1d(z)
x_trend = np.linspace(0, 100, 100)
ax1.plot(x_trend, p(x_trend), 'r--', alpha=0.5, label=f'Trend (slope={z[0]:.2f})')
ax1.legend()

# Plot 2: Within-occupation gaps
ax2 = axes[1]
colors = ['darkred' if g < -0.15 else 'red' if g < -0.10 else 'orange' if g < -0.05 else 'green' 
          for g in occ_df['Within_Gap']]

bars = ax2.barh(occ_df['Occupation'], occ_df['Within_Gap'] * 100, color=colors, edgecolor='black')
ax2.axvline(x=0, color='black', linewidth=1)
ax2.axvline(x=raw_gap * 100, color='blue', linestyle='--', linewidth=2, 
            label=f'Overall Gap ({raw_gap*100:.1f}%)')
ax2.set_xlabel('Within-Occupation Gender Gap (%)')
ax2.set_title('Gender Gap Within Each Occupation')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.savefig('reports/figures/occupation_segregation_gaps.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
correlation = occ_df['Female_Share'].corr(occ_df['Avg_Wage'])
print(f"\n📊 Correlation between female share and average wage: {correlation:.3f}")
print(f"   {'Higher female share → Lower wages' if correlation < 0 else 'No clear pattern'}")

## 5. Full Intersectional Heatmap

Visualizing wage gaps across all Gender × Immigration × Province combinations.

In [ ]:
# ============================================================================
# FULL INTERSECTIONAL ANALYSIS: GENDER × IMMIG × PROVINCE
# ============================================================================

# Calculate gaps for all combinations
intersections = []

# Reference: Canadian-born Male in each province
for prov in df['PROV'].unique():
    if prov not in prov_labels:
        continue
    
    prov_data = df[df['PROV'] == prov]
    
    # Get reference (Canadian-born male)
    ref_data = prov_data[(prov_data['IS_FEMALE'] == 0) & (prov_data['IMMIG'] == 1)]
    if len(ref_data) < 50:
        continue
    ref_wage = np.average(ref_data['REAL_HRLYEARN'], weights=ref_data['FINALWT'])
    
    for gender in [0, 1]:
        for immig in [1, 2, 3]:
            subset = prov_data[(prov_data['IS_FEMALE'] == gender) & (prov_data['IMMIG'] == immig)]
            if len(subset) < 50:
                continue
            
            wage = np.average(subset['REAL_HRLYEARN'], weights=subset['FINALWT'])
            gap = (wage - ref_wage) / ref_wage
            
            intersections.append({
                'Province': prov_labels[prov],
                'Gender': gender_labels[gender],
                'Immigration': immig_labels.get(immig, 'Unknown'),
                'Wage': wage,
                'Gap': gap,
                'N': len(subset)
            })

intersect_df = pd.DataFrame(intersections)

# Create combined group label
intersect_df['Group'] = intersect_df['Gender'] + ' × ' + intersect_df['Immigration']

print("=" * 70)
print("FULL INTERSECTIONAL ANALYSIS")
print("=" * 70)
print(f"\nTotal intersection cells: {len(intersect_df)}")
print(f"Largest penalty: {intersect_df['Gap'].min():.1%}")
print(f"Largest premium: {intersect_df['Gap'].max():.1%}")

In [ ]:
# ============================================================================
# INTERSECTIONAL HEATMAP
# ============================================================================

# Pivot for heatmap
heatmap_data = intersect_df.pivot_table(
    index='Group',
    columns='Province',
    values='Gap',
    aggfunc='mean'
) * 100  # Convert to percentage

# Reorder groups logically
group_order = [
    'Male × Canadian-born',
    'Male × Immigrant 10+',
    'Male × Recent Immig <10',
    'Female × Canadian-born',
    'Female × Immigrant 10+',
    'Female × Recent Immig <10'
]
heatmap_data = heatmap_data.reindex([g for g in group_order if g in heatmap_data.index])

# Province order (East to West)
prov_order = ['NL', 'PE', 'NS', 'NB', 'QC', 'ON', 'MB', 'SK', 'AB', 'BC']
heatmap_data = heatmap_data[[p for p in prov_order if p in heatmap_data.columns]]

# Create heatmap
fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn_r',
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Wage Gap vs Canadian-born Male (%)'},
    ax=ax
)

ax.set_title('Intersectional Wage Gaps: Gender × Immigration × Province\n'
             '(Reference: Canadian-born Male in each province = 0%)', fontsize=12)
ax.set_xlabel('Province (East to West)')
ax.set_ylabel('')

# Add horizontal line to separate male/female groups
ax.axhline(y=3, color='black', linewidth=2)

plt.tight_layout()
plt.savefig('reports/figures/intersectional_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💾 Figure saved to reports/figures/intersectional_heatmap.png")

## 6. Statistical Tests for Intersection Effects

Testing whether intersection terms are statistically significant beyond main effects.

In [ ]:
# ============================================================================
# REGRESSION ANALYSIS WITH INTERACTIONS
# ============================================================================

from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# Prepare data for regression
df_reg = df.dropna(subset=['LOG_HRLYEARN', 'IS_FEMALE', 'IS_IMMIGRANT', 'AGE', 'EDUC', 'PROV', 'NOC_10']).copy()

# Create interaction terms
df_reg['FEMALE_X_IMMIG'] = df_reg['IS_FEMALE'] * df_reg['IS_IMMIGRANT']

# Create dummy variables for provinces (reference: Ontario)
for prov in [10, 12, 24, 46, 47, 48, 59]:  # Skip small provinces
    df_reg[f'PROV_{prov}'] = (df_reg['PROV'] == prov).astype(int)
    df_reg[f'FEMALE_X_PROV_{prov}'] = df_reg['IS_FEMALE'] * df_reg[f'PROV_{prov}']

# Regression model
X_vars = ['IS_FEMALE', 'IS_IMMIGRANT', 'FEMALE_X_IMMIG', 'AGE', 'EDUC']
X_vars += [f'PROV_{p}' for p in [10, 12, 24, 46, 47, 48, 59]]
X_vars += [f'FEMALE_X_PROV_{p}' for p in [10, 12, 24, 46, 47, 48, 59]]

X = df_reg[X_vars]
X = sm.add_constant(X)
y = df_reg['LOG_HRLYEARN']
weights = df_reg['FINALWT']

# Weighted OLS
model = sm.WLS(y, X, weights=weights).fit()

print("=" * 70)
print("REGRESSION WITH INTERACTION TERMS")
print("=" * 70)
print("\nDependent Variable: Log(Real Hourly Earnings)")
print(f"N = {len(df_reg):,}, R² = {model.rsquared:.4f}")
print("\n--- Main Effects ---")
print(f"  Female:         {model.params['IS_FEMALE']:.4f} ({model.pvalues['IS_FEMALE']:.4f})")
print(f"  Immigrant:      {model.params['IS_IMMIGRANT']:.4f} ({model.pvalues['IS_IMMIGRANT']:.4f})")
print("\n--- Interaction Effect ---")
print(f"  Female × Immig: {model.params['FEMALE_X_IMMIG']:.4f} ({model.pvalues['FEMALE_X_IMMIG']:.4f})")

# Test joint significance of interactions
interaction_vars = ['FEMALE_X_IMMIG'] + [f'FEMALE_X_PROV_{p}' for p in [10, 12, 24, 46, 47, 48, 59]]
r_matrix = np.zeros((len(interaction_vars), len(model.params)))
for i, var in enumerate(interaction_vars):
    r_matrix[i, model.params.index.get_loc(var)] = 1

f_test = model.f_test(r_matrix)
print(f"\n--- Joint Test of All Interactions ---")
print(f"  F-statistic: {f_test.fvalue[0][0]:.2f}")
print(f"  p-value:     {f_test.pvalue:.4f}")
print(f"  Result:      {'Interactions are jointly significant ✓' if f_test.pvalue < 0.05 else 'Not significant'}")

## 7. Summary and Key Findings

In [ ]:
# ============================================================================
# SUMMARY OF INTERSECTIONAL FINDINGS
# ============================================================================

print("=" * 70)
print("INTERSECTIONAL ANALYSIS - KEY FINDINGS")
print("=" * 70)

print("\n📊 BASELINE")
print(f"   Overall gender wage gap: {raw_gap:.1%}")

print("\n👥 GENDER × IMMIGRATION")
print(f"   Female penalty:          {female_effect:.1%}")
print(f"   Immigrant penalty:       {immigrant_effect:.1%}")
print(f"   Compound penalty:        {compound_penalty:.1%}")
if compound_penalty < -0.01:
    print("   → Double jeopardy confirmed: Immigrant women face extra penalty")

print("\n🗺️ GENDER × PROVINCE")
print(f"   Province with largest gap:  {prov_df.iloc[0]['Province']} ({prov_df.iloc[0]['Gap']:.1%})")
print(f"   Province with smallest gap: {prov_df.iloc[-1]['Province']} ({prov_df.iloc[-1]['Gap']:.1%})")

print("\n💼 GENDER × OCCUPATION")
print(f"   Female share-wage correlation: {correlation:.3f}")
print(f"   Occupation with largest gap:   {occ_df.iloc[0]['Occupation']} ({occ_df.iloc[0]['Within_Gap']:.1%})")
print(f"   Occupation with smallest gap:  {occ_df.iloc[-1]['Occupation']} ({occ_df.iloc[-1]['Within_Gap']:.1%})")

print("\n📈 STATISTICAL SIGNIFICANCE")
print(f"   Female × Immigrant interaction: p = {model.pvalues['FEMALE_X_IMMIG']:.4f}")
print(f"   Joint test of all interactions: p = {f_test.pvalue:.4f}")

print("\n" + "=" * 70)
print("POLICY IMPLICATIONS")
print("=" * 70)
print("""
1. Pay equity policies should consider intersectionality:
   - Immigrant women face compounding disadvantages
   - One-size-fits-all gender policies may be insufficient

2. Provincial variation suggests local policy matters:
   - Provinces with pay equity legislation show different patterns
   - Policy learning across provinces could help

3. Occupational segregation is a major driver:
   - Female-dominated occupations pay less
   - Within-occupation gaps persist
   - Both horizontal and vertical segregation matter

4. Targeted interventions needed:
   - Focus on most disadvantaged intersections
   - Immigrant women in low-wage provinces/occupations
""")

In [ ]:
# ============================================================================
# SAVE RESULTS
# ============================================================================

import os

# Create reports directory if needed
os.makedirs('reports/figures', exist_ok=True)

# Save intersection data
intersect_df.to_csv('reports/intersectional_gaps.csv', index=False)
prov_df.to_csv('reports/provincial_gaps.csv', index=False)
occ_df.to_csv('reports/occupational_gaps.csv', index=False)

print("\n💾 Results saved:")
print("   - reports/intersectional_gaps.csv")
print("   - reports/provincial_gaps.csv")
print("   - reports/occupational_gaps.csv")
print("   - reports/figures/gender_immigration_intersection.png")
print("   - reports/figures/provincial_gender_gaps.png")
print("   - reports/figures/occupation_segregation_gaps.png")
print("   - reports/figures/intersectional_heatmap.png")